# My seq2seq

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.conv = nn.Conv2d(in_channels=input_dim[2], out_channels=16, kernel_size=(3, 3))
        self.inception = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding=1),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), padding=1)
        )
        # Compute the size after conv and inception layers
        self.conv_output_size = self._get_conv_output_size(input_dim)
        self.lstm = nn.LSTM(input_size=self.conv_output_size, hidden_size=hidden_dim, batch_first=True)

    def _get_conv_output_size(self, input_dim):
        with torch.no_grad():
            dummy_input = torch.zeros(1, input_dim[2], input_dim[0], input_dim[1])
            dummy_output = self.inception(F.relu(self.conv(dummy_input)))
            return int(dummy_output.numel() / dummy_output.size(0))

    def forward(self, x):
        x = F.relu(self.conv(x))
        x = self.inception(x)
        # Reshape for LSTM input
        batch_size = x.size(0)
        x = x.view(batch_size, -1, self.conv_output_size)  # Flatten the spatial dimensions
        output, hidden = self.lstm(x)
        return output, hidden



In [3]:
# Define input dimensions and hidden dimensions
input_dim = (100, 100, 3)  # Example: (height, width, channels)
hidden_dim = 128

# Instantiate the encoder
encoder = Encoder(input_dim=input_dim, hidden_dim=hidden_dim)

# Create a random input tensor with shape (batch_size, channels, height, width)
batch_size = 2
input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])

# Forward pass through the encoder
output, hidden = encoder(input_tensor)

# Print the shapes of the output and hidden state
print("Output shape:", output.shape)
print("Hidden shape:", hidden[0].shape)  # hidden[0] is the hidden state
print("Cell state shape:", hidden[1].shape)  # hidden[1] is the cell state

Output shape: torch.Size([2, 1, 128])
Hidden shape: torch.Size([1, 2, 128])
Cell state shape: torch.Size([1, 2, 128])


In [4]:
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.lstm = nn.LSTM(input_size=output_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden):
        output, hidden = self.lstm(x, hidden)
        output = self.fc(output)
        return output, hidden


# Define input dimensions and hidden dimensions for the encoder and decoder
input_dim = (100, 100, 3)  # Example: (height, width, channels)
hidden_dim = 128

# Instantiate the encoder and decoder
encoder = Encoder(input_dim=input_dim, hidden_dim=hidden_dim)
decoder = Decoder(output_dim=encoder.conv_output_size, hidden_dim=hidden_dim)

# Create a random input tensor with shape (batch_size, channels, height, width)
batch_size = 2
input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])

# Forward pass through the encoder
encoder_output, encoder_hidden = encoder(input_tensor)

# Prepare initial input for the decoder (e.g., a sequence of zeros)
decoder_input = torch.zeros(batch_size, 1, encoder.conv_output_size)

# Forward pass through the decoder
decoder_output, decoder_hidden = decoder(decoder_input, encoder_hidden)

# Print the shapes of the output and hidden state
print("Encoder output shape:", encoder_output.shape)
print("Encoder hidden shape:", encoder_hidden[0].shape, encoder_hidden[1].shape)
print("Decoder output shape:", decoder_output.shape)
print("Decoder hidden shape:", decoder_hidden[0].shape, decoder_hidden[1].shape)


Encoder output shape: torch.Size([2, 1, 128])
Encoder hidden shape: torch.Size([1, 2, 128]) torch.Size([1, 2, 128])
Decoder output shape: torch.Size([2, 1, 614656])
Decoder hidden shape: torch.Size([1, 2, 128]) torch.Size([1, 2, 128])


In [5]:

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_tensor):
        enc_output, enc_hidden = self.encoder(input_tensor)
        
        # Initialize decoder's input with a tensor of zeros
        decoder_input = torch.zeros(input_tensor.size(0), 1, self.decoder.fc.out_features)
        decoder_input = decoder_input.to(input_tensor.device)
        
        dec_output, dec_hidden = self.decoder(decoder_input, enc_hidden)
        
        return dec_output

# Example usage

# Define input dimensions and hidden dimensions for the encoder and decoder
input_dim = (100, 100, 3)  # Example: (height, width, channels)
hidden_dim = 128
output_dim = 64 * 25  # This should match the encoder's conv output size
# Instantiate the encoder, decoder, and seq2seq model
encoder = Encoder(input_dim=input_dim, hidden_dim=hidden_dim)
decoder = Decoder(output_dim=encoder.conv_output_size, hidden_dim=hidden_dim)
seq2seq = Seq2Seq(encoder, decoder)

# Create a random input tensor with shape (batch_size, channels, height, width)
batch_size = 2
input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])

# Forward pass through the seq2seq model
output = seq2seq(input_tensor)

# Print the shape of the output
print("Seq2Seq output shape:", output.shape)

Seq2Seq output shape: torch.Size([2, 1, 614656])


# Multi-horizon 

The 2 ddifferent things from the paper are within the definition of the encoder. 
- the padding is specified each layer by hand (e.g. `padding=(2, 0)`, while in keras it was `padding="same"`)
- with this padding there was a dimensional mismatch in the lstm layer so i adjusted like this 
  ```python
  self.lstm = nn.LSTM(input_size=192, hidden_size=latent_dim, batch_first=True) # Adjusted LSTM input size
  ```
  while before it was 
  ```python
  self.encoder_lstm = nn.LSTM(input_size=64, hidden_size=latent_dim, batch_first=True)
  ```

In [6]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=input_dim[2], out_channels=32, kernel_size=(1, 2), stride=(1, 2))
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0)) # Adjusted padding
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1, 2), stride=(1, 2))
        self.conv5 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))
        self.conv6 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv7 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(1, 10))
        self.conv8 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))
        self.conv9 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(4, 1), padding=(2, 0))

        self.conv10 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))
        self.conv11 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 1), padding=(1, 0))

        self.conv12 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))
        self.conv13 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(5, 1), padding=(2, 0))

        self.pool = nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        self.conv14 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(1, 1), padding=(0, 0))

        self.lstm = nn.LSTM(input_size=192, hidden_size=latent_dim, batch_first=True) # Adjusted LSTM input size

    def forward(self, x):
        x = F.leaky_relu(self.conv1(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv2(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv3(x), negative_slope=0.01)

        x = F.leaky_relu(self.conv4(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv5(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv6(x), negative_slope=0.01)

        x = F.leaky_relu(self.conv7(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv8(x), negative_slope=0.01)
        x = F.leaky_relu(self.conv9(x), negative_slope=0.01)

        convsecond_1 = F.leaky_relu(self.conv10(x), negative_slope=0.01)
        convsecond_1 = F.leaky_relu(self.conv11(convsecond_1), negative_slope=0.01)

        convsecond_2 = F.leaky_relu(self.conv12(x), negative_slope=0.01)
        convsecond_2 = F.leaky_relu(self.conv13(convsecond_2), negative_slope=0.01)

        convsecond_3 = self.pool(x)
        convsecond_3 = F.leaky_relu(self.conv14(convsecond_3), negative_slope=0.01)

        x = torch.cat([convsecond_1, convsecond_2, convsecond_3], dim=1)
        x = x.view(x.size(0), x.size(2), x.size(1))  # Reshape for LSTM

        output, (state_h, state_c) = self.lstm(x)
        return output, state_h, state_c

In [7]:
# Example usage of the Encoder
if __name__ == "__main__":
    # Define input dimensions and latent dimensions
    input_dim = (50, 40, 1)  # Example: (height, width, channels)
    latent_dim = 128

    # Instantiate the encoder
    encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
    
    # Create a random input tensor with shape (batch_size, channels, height, width)
    batch_size = 2
    input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])
    
    # Forward pass through the encoder
    output, state_h, state_c = encoder(input_tensor)
    
    # Print the shapes of the output and hidden states
    print("Encoder output shape:", output.shape)
    print("Encoder hidden state (h) shape:", state_h.shape)
    print("Encoder cell state (c) shape:", state_c.shape)

Encoder output shape: torch.Size([2, 56, 128])
Encoder hidden state (h) shape: torch.Size([1, 2, 128])
Encoder cell state (c) shape: torch.Size([1, 2, 128])


In [8]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.lstm = nn.LSTM(input_size=latent_dim + 3, hidden_size=latent_dim, batch_first=True)
        self.fc = nn.Linear(latent_dim, 3)

    def forward(self, x, hidden):
        lstm_out, hidden = self.lstm(x, hidden)
        output = self.fc(lstm_out)
        return output, hidden

In [9]:
# Example usage of the Decoder
if __name__ == "__main__":
    # Define latent dimensions and sequence length
    latent_dim = 128
    seq_len = 5
    output_dim = 3

    # Instantiate the decoder
    decoder = Decoder(latent_dim=latent_dim)

    # Create a random decoder input tensor with shape (batch_size, sequence_length, output_dim)
    batch_size = 2
    decoder_input = torch.randn(batch_size, 1, output_dim)

    # Create a random initial hidden state and cell state from the encoder
    h0 = torch.randn(1, batch_size, latent_dim)
    c0 = torch.randn(1, batch_size, latent_dim)
    hidden = (h0, c0)

    # Forward pass through the decoder for each time step
    all_outputs = []
    for _ in range(seq_len):
        decoder_input = torch.cat([decoder_input, h0.permute(1, 0, 2)], dim=-1)  # Concatenate with hidden state
        output, hidden = decoder(decoder_input, hidden)
        all_outputs.append(output)
        decoder_input = output  # Use the output as the next input

    # Concatenate all outputs
    final_output = torch.cat(all_outputs, dim=1)

    # Print the shape of the final decoder output
    print("Final decoder output shape:", final_output.shape)


Final decoder output shape: torch.Size([2, 5, 3])


True seq 2 seq

In [10]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_tensor, target_length):
        enc_output, state_h, state_c = self.encoder(input_tensor)
        hidden = (state_h, state_c)
        
        # Initialize decoder's input with a tensor of zeros
        decoder_input = torch.zeros(input_tensor.size(0), 1, 3, device=input_tensor.device)
        decoder_input = torch.cat([decoder_input, enc_output[:, -1:, :]], dim=2)
        
        all_outputs = []
        
        for _ in range(target_length):
            output, hidden = self.decoder(decoder_input, hidden)
            all_outputs.append(output)
            decoder_input = torch.cat([output, enc_output[:, -1:, :]], dim=2)
        
        decoder_outputs = torch.cat(all_outputs, dim=1)
        return decoder_outputs
    
    

In [11]:
# Example usage of the Seq2Seq model
if __name__ == "__main__":
    # Define input dimensions and latent dimensions
    input_dim = (50, 40, 1)  # Example: (height, width, channels)
    latent_dim = 128

    # Instantiate the encoder and decoder
    encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim)
    decoder = Decoder(latent_dim=latent_dim)

    # Instantiate the Seq2Seq model
    seq2seq = Seq2Seq(encoder, decoder)
    
    # Create a random input tensor with shape (batch_size, channels, height, width)
    batch_size = 2
    input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])
    
    # Define the target length for the output sequence
    target_length = 5
    
    # Forward pass through the Seq2Seq model
    output = seq2seq(input_tensor, target_length)
    
    # Print the shape of the final output
    print("Seq2Seq model output shape:", output.shape)

Seq2Seq model output shape: torch.Size([2, 5, 3])


In [12]:
# optimize the model parameters

# Define the loss function and optimizer
criterion = nn.MSELoss()

# Define the input dimensions and latent dimensions
input_dim = (50, 40, 1)  # Example: (height, width, channels)

# Instantiate the encoder and decoder
encoder = Encoder(input_dim=input_dim, latent_dim=128)
decoder = Decoder(latent_dim=128)

# Instantiate the Seq2Seq model
seq2seq = Seq2Seq(encoder, decoder)

# Create a random input tensor with shape (batch_size, channels, height, width)
batch_size = 2
input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])

# Define the target length for the output sequence
target_length = 5

print("input_tensor", input_tensor.shape)
print("target_length", target_length)


# Forward pass through the Seq2Seq model
output = seq2seq(input_tensor, target_length)

# Create a random target tensor with the same shape as the output
target = torch.randn(batch_size, target_length, 3)

# Calculate the loss
loss = criterion(output, target)

print("target shape", target.shape)

# Print the loss
print("Loss:", loss.item())

# Perform backpropagation
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=0.001)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Print the updated parameters
for name, param in seq2seq.named_parameters():
    if param.requires_grad:
        print(name, param.data)



input_tensor torch.Size([2, 1, 50, 40])
target_length 5


target shape torch.Size([2, 5, 3])
Loss: 1.2680433988571167
encoder.conv1.weight tensor([[[[-0.1331,  0.6495]]],


        [[[ 0.0439,  0.3926]]],


        [[[ 0.2886, -0.1574]]],


        [[[ 0.4715, -0.2036]]],


        [[[-0.0732, -0.6024]]],


        [[[ 0.4950,  0.5616]]],


        [[[ 0.2902, -0.5231]]],


        [[[-0.2686, -0.3127]]],


        [[[-0.3218, -0.3150]]],


        [[[-0.6191,  0.2512]]],


        [[[-0.3167,  0.3663]]],


        [[[-0.6906, -0.5133]]],


        [[[ 0.3752, -0.2836]]],


        [[[ 0.1418,  0.0391]]],


        [[[-0.0968, -0.1231]]],


        [[[ 0.1935,  0.1150]]],


        [[[-0.6450,  0.1497]]],


        [[[ 0.1153,  0.1159]]],


        [[[ 0.1408, -0.4050]]],


        [[[-0.0629,  0.5413]]],


        [[[ 0.4970,  0.2413]]],


        [[[-0.3585, -0.0836]]],


        [[[ 0.3180,  0.4595]]],


        [[[-0.2268, -0.3707]]],


        [[[-0.3203, -0.0233]]],


        [[[ 0.5886, -0.3349]]],


        [[[-0.6841,  0.1638]]],


 

In [17]:
# Define the number of epochs
num_epochs = 100

# Define the input dimensions and latent dimensions
input_dim = (50, 40, 1)  # Example: (height, width, channels)

n_hidden = 64

# Check the dim
print(input_dim)
print(n_hidden) # latent_dim
print()

# Instantiate the encoder and decoder
encoder = Encoder(input_dim=input_dim, latent_dim=n_hidden)
decoder = Decoder(latent_dim=n_hidden)

# Instantiate the Seq2Seq model
seq2seq = Seq2Seq(encoder, decoder)

# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=0.001)

# Generate random input and target tensors
batch_size = 2
input_tensor = torch.randn(batch_size, input_dim[2], input_dim[0], input_dim[1])
target_length = 5
target = torch.randn(batch_size, target_length, 3)

# Check the dim
print(input_dim)
print(n_hidden) # latent_dim
print()

# Training loop
for epoch in range(num_epochs):
    optimizer.zero_grad()
    print(input_tensor.shape)
    print(target_length)
    output = seq2seq(input_tensor, target_length)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")



(50, 40, 1)
64

(50, 40, 1)
64

torch.Size([2, 1, 50, 40])
5
Epoch 1, Loss: 1.7927730083465576
torch.Size([2, 1, 50, 40])
5
Epoch 2, Loss: 1.7696088552474976
torch.Size([2, 1, 50, 40])
5
Epoch 3, Loss: 1.746444821357727
torch.Size([2, 1, 50, 40])
5
Epoch 4, Loss: 1.7225137948989868
torch.Size([2, 1, 50, 40])
5
Epoch 5, Loss: 1.697058081626892
torch.Size([2, 1, 50, 40])
5
Epoch 6, Loss: 1.6692249774932861
torch.Size([2, 1, 50, 40])
5
Epoch 7, Loss: 1.6382930278778076
torch.Size([2, 1, 50, 40])
5
Epoch 8, Loss: 1.6031385660171509
torch.Size([2, 1, 50, 40])
5
Epoch 9, Loss: 1.5624268054962158
torch.Size([2, 1, 50, 40])
5
Epoch 10, Loss: 1.516164779663086
torch.Size([2, 1, 50, 40])
5
Epoch 11, Loss: 1.4652519226074219
torch.Size([2, 1, 50, 40])
5
Epoch 12, Loss: 1.4166959524154663
torch.Size([2, 1, 50, 40])
5
Epoch 13, Loss: 1.3872030973434448
torch.Size([2, 1, 50, 40])
5
Epoch 14, Loss: 1.3979146480560303
torch.Size([2, 1, 50, 40])
5
Epoch 15, Loss: 1.4153047800064087
torch.Size([2, 1, 50